# 🎙️ Stable Money AI Avatar Agent
**Run this notebook on Google Colab with T4 GPU (free)**

This will:
1. Install all dependencies
2. Download AI models
3. Start your avatar server
4. Give you a public URL via ngrok

⚠️ Make sure Runtime → Change runtime type → T4 GPU is selected before running!

In [ ]:
# ── CELL 1: Check GPU ──
import torch
print('GPU available:', torch.cuda.is_available())
print('GPU name:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None')
!nvidia-smi

In [ ]:
!pip install -q fastapi "uvicorn[standard]" python-multipart aiofiles
!pip install -q kokoro>=0.9.2 soundfile
!apt-get -qq install -y espeak-ng
!pip install -q pyngrok
!pip install -q nest_asyncio
# Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh
print('✅ Dependencies installed')


In [ ]:
# ── CELL 3: Install Wav2Lip ──
import os

if not os.path.exists('/content/Wav2Lip'):
    !git clone https://github.com/Rudrabha/Wav2Lip.git /content/Wav2Lip
    !pip install -q -r /content/Wav2Lip/requirements.txt
    # Download Wav2Lip GAN checkpoint
    !pip install -q gdown
    import gdown
    os.makedirs('/content/checkpoints', exist_ok=True)
    gdown.download(
        'https://drive.google.com/uc?id=1j4BgkHOUFNYYFyHMVJ0Dq1Y6Vo5mLyBn',
        '/content/checkpoints/wav2lip_gan.pth', quiet=False
    )
    print('✅ Wav2Lip installed')
else:
    print('✅ Wav2Lip already installed')

In [ ]:
# ── CELL 4: Start Ollama + pull model ──
import subprocess, time, urllib.request, json

# Start Ollama server in background
subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(3)

# Pull fast model
print('Pulling llama3.2:3b — this takes 2-3 minutes...')
!ollama pull llama3.2:3b
print('✅ Ollama ready')

In [ ]:
server_code = '''
import asyncio, base64, io, os, sys
from pathlib import Path
from typing import Optional
import torch
from fastapi import FastAPI, WebSocket, WebSocketDisconnect, UploadFile, File
from fastapi.middleware.cors import CORSMiddleware
from fastapi.staticfiles import StaticFiles
from kokoro import KPipeline
import soundfile as sf
import numpy as np

app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
VOICE_SAMPLE = "/content/voice_sample.wav"
print(f"[server] Device: {DEVICE}")

class Models:
    tts = None
models = Models()

@app.on_event("startup")
async def load_models():
    models.tts = KPipeline(lang_code='a')
    print("[startup] Kokoro TTS loaded")

def synth(text):
    generator = models.tts(text, voice='af_heart', speed=1.1)
    chunks = []
    for _, _, audio in generator:
        chunks.append(audio)
    audio_data = np.concatenate(chunks)
    buf = io.BytesIO()
    sf.write(buf, audio_data, 24000, format='WAV')
    buf.seek(0)
    return buf.read()

@app.websocket("/ws/talk")
async def ws_talk(ws: WebSocket):
    await ws.accept()
    try:
        while True:
            data = await ws.receive_json()
            if data.get("type") == "config": continue
            if data.get("type") != "speak": continue
            text = data.get("text","").strip()
            if not text: continue
            prompt = f"""You are Aayush, a warm advisor for Stable Money India.
Stable Money offers FDs from 25+ NBFCs with 8-9.5% returns. Sukoon: no-lock-in 7-8%. Min Rs 1000. DICGC insured up to Rs 5L.
Answer in 2 sentences max. Warm and simple. No links.
Question: {text}"""
            try:
                import urllib.request as ur, json
                req = ur.Request("http://localhost:11434/api/generate",
                    data=json.dumps({"model":"llama3.2:3b","prompt":prompt,"stream":False,
                        "options":{"num_predict":60,"temperature":0.7}}).encode(),
                    headers={"Content-Type":"application/json"})
                with ur.urlopen(req, timeout=30) as r:
                    answer = json.loads(r.read()).get("response","").strip()
            except Exception as e:
                answer = f"Sorry, error: {e}"
            await ws.send_json({"type":"text_chunk","text":answer})
            try:
                audio = await asyncio.get_event_loop().run_in_executor(None, synth, answer)
                await ws.send_json({"type":"audio","data":base64.b64encode(audio).decode(),"format":"wav"})
            except Exception as e:
                await ws.send_json({"type":"error","msg":str(e)})
            await ws.send_json({"type":"done"})
    except WebSocketDisconnect:
        pass

@app.post("/upload/voice")
async def upload_voice(file: UploadFile = File(...)):
    with open(VOICE_SAMPLE,"wb") as f: f.write(await file.read())
    return {"status":"ok"}

@app.get("/health")
def health():
    return {"status":"ok","device":DEVICE}

if Path("./static").exists():
    app.mount("/", StaticFiles(directory="static",html=True), name="static")
'''
with open('/content/server.py', 'w') as f:
    f.write(server_code)
print('✅ server.py written')


In [ ]:
# ── CELL 6: Clone frontend from GitHub ──
import os
os.makedirs('/content/static', exist_ok=True)

!curl -s https://raw.githubusercontent.com/aayushjha-blip/stable-money-avatar/main/static/index.html -o /content/static/index.html

# Verify
size = os.path.getsize('/content/static/index.html')
print(f'✅ Frontend downloaded ({size} bytes)')

In [ ]:
# ── CELL 7: Upload your avatar photo (run this cell) ──
from google.colab import files
print('Upload your front-facing photo:')
uploaded = files.upload()
for name, data in uploaded.items():
    with open('/content/avatar.jpg', 'wb') as f:
        f.write(data)
    print(f'✅ Avatar saved: {name} ({len(data)} bytes)')

In [ ]:
# ── CELL 8: Upload voice sample (optional but recommended) ──
from google.colab import files
print('Upload your voice_sample.wav (30s recording):')
uploaded = files.upload()
for name, data in uploaded.items():
    with open('/content/voice_sample.wav', 'wb') as f:
        f.write(data)
    print(f'✅ Voice sample saved: {name} ({len(data)} bytes)')

In [ ]:
from pyngrok import conf
NGROK_TOKEN = "YOUR_NGROK_TOKEN_HERE"  # ← paste your token
conf.get_default().auth_token = NGROK_TOKEN
!ngrok config add-authtoken {NGROK_TOKEN}
print('✅ ngrok configured')


In [ ]:
import nest_asyncio, uvicorn, os, sys
from pyngrok import ngrok
nest_asyncio.apply()
os.chdir('/content')
sys.path.insert(0, '/content')

# Kill old tunnels
ngrok.kill()
import time; time.sleep(2)

tunnel = ngrok.connect(8000)
public_url = tunnel.public_url
ws_url = public_url.replace('https://', 'wss://') + '/ws/talk'
print('=' * 50)
print('🌐 PUBLIC URL:', public_url)
print('🔌 WS URL:', ws_url)
print('=' * 50)

config = uvicorn.Config('server:app', host='0.0.0.0', port=8000, log_level='info')
server_inst = uvicorn.Server(config)
await server_inst.serve()
